In [ ]:

import pandas as pd
import sys
project_root = "/data/workdata/709656/Anne/Emilys_code/"
sys.path.insert(0,project_root)
print(sys.path[0])
from preprocessing.split_format3 import split_and_format_data
from models.xgboost_anne4 import train_xgboost_model_random

In [ ]:
DATA_PATH = ("/Data_files/cohort_data/cohort_50_to_54.parquet" )
RANDOM_STATE=42
MODEL_NAME="Model5"
MAXIMIZE_METRIC = "f2" # or pr_auc, recall, precision, roc_auc
#MIN_PRECISION =0.15

In [ ]:
from pathlib import Path
COHORT_NAME= "male_50-54"
OUTPUT_DIR = Path(
    f"/XBoost_results/{COHORT_NAME}"
)
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)

In [ ]:
import fastparquet
# Load, split, preprocess
(
X_train_raw,
X_cal_raw,
X_test_raw,
y_train,
y_cal,
y_test,
id_fit,
if_cal,
id_test,
preprocessor
) = split_and_format_data(
    data_path=DATA_PATH,
    drop_cols=[
        "pnr",
        "family_id",
        "in_dk",
        "de_age",
        "alive",
        "de_parish",
        "de_region",
        "de_municipality",
        "de_time_to_death",
        "de_age_at_death",
        "se_educ_date",
        "de_sex",
    ],
sex_filter = ["Male"],
target_col="early_death",
test_size=0.3,
cal_size_within_train=0.3,
random_state=RANDOM_STATE,
stratify_on_year=True,
year_col="year",
)
print("Train shape:", X_train_raw.shape)
print("Cal shape:", X_cal_raw.shape)
print("Test shape:", X_test_raw.shape)
print("Train death rate:", y_train.mean())
print("Cal death rate:", y_cal.mean())
print("Test death rate:", y_test.mean())

In [ ]:
# death rates
import numpy as np
raw_train_death_rate = float(np.mean(y_train))
cal_death_rate = float(np.mean(y_cal))
test_death_rate =float(np.mean(y_test))

In [ ]:
#Scaleposweight
spw= (1-y_train.mean()) /y_train.mean()
print(spw)

In [ ]:
#Hyperparameters
param_grid = {
    "max_depth": [2,3,4],
    "learning_rate": [0.01,0.03,0.05],
    "n_estimators": [400,800,1000],
    "subsample": [0.5, 0.6, 0.7],
    "colsample_bytree": [0.3, 0.5, 0.7],
    "gamma":[2,5,10,20],
    "min_child_weight":[10,20,40],
    "reg_lambda":[20,40,80],
    "reg_alpha":[1,5,10],
}

In [ ]:
search_seed = abs(hash(COHORT_NAME)) % (2**32)
model5_best_model, model_5_best_params,model5_thr = train_xgboost_model_random(
    X_train_raw,
    y_train,
    preprocessor=preprocessor,
    param_grid=param_grid,
    cv_folds=3,
    random_state=search_seed,
    maximize="f2",
    #min_precision=0.15,
    scale_pos_weight=spw,
    n_iter=60, )

In [ ]:
from sklearn.calibration import CalibratedClassifierCV
calibrated_model = CalibratedClassifierCV(
    estimator=model5_best_model,
    method="isotonic",
    cv= "prefit"
)
calibrated_model.fit(X_cal_raw,y_cal)

In [ ]:
## Predicted probability for survivors vs deaths + gap. Calibrated model.
    # split data by outcome
X_survivors = X_test_raw[y_test == 0].copy()
X_deaths = X_test_raw[y_test == 1].copy()
#### OBS model
    #  gap
p_survivors_cal = calibrated_model.predict_proba(X_survivors)[:, 1].mean()
p_deaths_cal = calibrated_model.predict_proba(X_deaths)[:, 1].mean()
total_gap_cal = p_deaths_cal - p_survivors_cal
print("=== Predicted probabilites on test Calibrated model ===")
print(f"Survivors (y=0), mean p(death): {p_survivors_cal:.4f}")
print(f"Early deaths (y=1), mean p(death): {p_deaths_cal:.4f}")
print(f"Mortality gap (death-survivors): {total_gap_cal:.4f}")

In [ ]:
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt
#Raw model
y_prob_raw = model5_best_model.predict_proba(X_test_raw)[:,1]
# Calibrated model
y_prob_cal = calibrated_model.predict_proba(X_test_raw)[:,1]
prob_true_raw, prob_pred_raw=calibration_curve(y_test,y_prob_raw, n_bins=10)
prob_true_cal, prob_pred_cal=calibration_curve(y_test,y_prob_cal, n_bins=10)
plt.figure(figsize=(6,6))
plt.plot(prob_pred_raw, prob_true_raw, marker="o", label="Raw model")
plt.plot(prob_pred_cal, prob_true_cal, marker="o", label="Calibrated model")
plt.plot([0,1],[0,1], linestyle="--", color="gray", label="Perfect ↪ calibration")
plt.xlabel("Predicted probability")
plt.ylabel("Observed probability")
plt.title("Calibration Curve")
plt.legend()
plt.savefig( OUTPUT_DIR/ f"calibration_curve_{COHORT_NAME}.png", dpi=300, bbox_inches="tight") 
plt.show()

In [ ]:
#Save calibrated model
import joblib

joblib.dump(calibrated_model,
OUTPUT_DIR / f"model5_calibrated_model_{COHORT_NAME}.joblib")

In [ ]:

X_cal_raw.to_parquet(
    OUTPUT_DIR/ f"X_cal_raw_{COHORT_NAME}.parquet", index=False
)
pd.DataFrame({"y_cal": y_cal}).to_parquet(
    OUTPUT_DIR / f"y_cal_raw_{COHORT_NAME}.parquet",
    index=False
)

In [ ]:
#Find optimal calibtaion threshold
from sklearn.metrics import precision_recall_curve
import numpy as np
y_prob_cal = calibrated_model.predict_proba(X_cal_raw)[:, 1]

precision_cal, recall_cal, thresholds_cal=precision_recall_curve(y_cal,y_prob_cal)

# Choose threshold that maximizes f2 on calibtaion set
beta=2
p_cal= precision_cal[:-1]
r_cal= recall_cal[:-1]
f2_cal = (1 + beta**2) * (p_cal *r_cal) / (beta**2 * p_cal + r_cal + 1e-12)
best_idx_cal = np.argmax(f2_cal)
model5_thr_cal = float(thresholds_cal[best_idx_cal])

print("Chosen threshold ( max f2):", model5_thr_cal)
print("Precision / Recall at chosen thr:", p_cal[best_idx_cal],
r_cal[best_idx_cal])
print("F2 at chosen thr:", f2_cal[best_idx_cal])

In [ ]:
# Save calibrated predection
from pathlib import Path
import pandas as pd
import numpy as np
y_prob_cal=calibrated_model.predict_proba(X_test_raw)[:,1]
y_pred_cal = (y_prob_cal >=model5_thr_cal).astype(int)
pred_cal_df = pd.DataFrame({
    "pnr":id_test.astype(str),
    "y_test":np.asarray(y_test).astype(int),
    "y_proba_cal":y_prob_cal,
    "y_pred":y_pred_cal,
    "celibrated_threshold":model5_thr_cal,
    "test_death_rate":y_test.mean()
})
pred_cal_df.to_parquet(OUTPUT_DIR/ "calibrated_predictions.parquet", index=False)

In [ ]:
# Save calibration data after preprocessing
preprocessor_fitted = model5_best_model.named_steps["preprocess"]
X_cal_processed = preprocessor_fitted.transform(X_cal_raw)
if hasattr(X_cal_processed, "toarray"): X_cal_processed = X_cal_processed.toarray()
X_cal_processed = pd.DataFrame(
    X_cal_processed,
    columns=preprocessor_fitted.get_feature_names_out()
)
X_cal_processed.columns = (
    X_cal_processed.columns
    .str.replace("^remainder__", "", regex=True)
    .str.replace("^cat__", "", regex=True)
)
X_cal_processed.to_csv(OUTPUT_DIR / f"X_cal_{COHORT_NAME}.csv", index=False)
pd.DataFrame({"y_cal": y_cal}).to_csv(OUTPUT_DIR /f"y_cal_{COHORT_NAME}.csv", index=False)


In [ ]:
import json
from pathlib import Path
from datetime import datetime
import numpy as np
from sklearn.metrics import(
    f1_score,
    precision_score,
    recall_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    accuracy_score,
    fbeta_score,
    roc_auc_score,
    matthews_corrcoef,
)

In [ ]:
# Evaluation on calibration set
y_prob_cal=calibrated_model.predict_proba(X_cal_raw)[:,1]
y_pred_cal = (y_prob_cal >=model5_thr_cal).astype(int)
cal_f1= f1_score(y_cal, y_pred_cal)
cal_f2 =fbeta_score(y_cal,y_pred_cal,beta=2)
cal_prec=precision_score(y_cal,y_pred_cal)
cal_rec=recall_score(y_cal,y_pred_cal)
cal_roc=roc_auc_score(y_cal,y_prob_cal)
cal_pr=average_precision_score(y_cal,y_prob_cal)
cal_bal=balanced_accuracy_score(y_cal,y_pred_cal)
cal_acc=accuracy_score(y_cal,y_pred_cal)
cal_mcc= matthews_corrcoef(y_cal, y_pred_cal)
print("\n ==== Calibration metric=====" )
print("F1 score:", cal_f1)
print("F2 score:", cal_f2)
print("Precision:", cal_prec)
print("Recall:", cal_rec)
print("ROC-AUC", cal_roc)
print("PR-AUC (avg prec):", cal_pr)
print("Balanced accuracy:", cal_bal)
print("Accuracy:", cal_acc)
print("MCC:", cal_mcc)

In [ ]:
from sklearn.metrics import confusion_matrix

tn_cal,fp_cal, fn_cal, tp_cal=confusion_matrix(y_cal,y_pred_cal).ravel()
cal_specificity = tn_cal/ (tn_cal+ fp_cal)
cal_fpr =fp_cal /(fp_cal+ tn_cal)

In [ ]:
# Save results
import json
import numpy as np
from datetime import datetime

time_stamp = datetime.now().strftime("%Y%m%d_%H%M%S")

results = {
    "model_name": "Model5_calibrated",

    "objective": {
        "cohort": f"{COHORT_NAME}",
        "maximize": "f2",
        "eval_metric": "aucpr",
        "min_precision": 0,
        "cv_folds": 3,
        "n_iter": 60,
        "scale_pos_weight": float(spw),
        "calibration_method": "isotonic",
        "comments": "XGBoost model trained with scale_pos_weight and calibrated with isotonic calibration"
    },

    "best_params": model_5_best_params,

    "death_rate_summary": {
        "raw_train_death_rate": float(raw_train_death_rate),
        "calibration_death_rate": float(cal_death_rate),
        "test_death_rate": float(test_death_rate),
    },

    "calibration_metrics": {
        "f1": float(cal_f1),
        "f2": float(cal_f2),
        "precision": float(cal_prec),
        "recall": float(cal_rec),
        "roc_auc": float(cal_roc),
        "pr_auc": float(cal_pr),
        "balanced_accuracy": float(cal_bal),
        "accuracy": float(cal_acc),
        "MCC": float(cal_mcc),
        "specificity": float(cal_specificity),
        "fpr": float(cal_fpr),
        "threshold_used": float(model5_thr_cal),
    },

    "confusion_matrix_calibration": {
        "TN": int(tn_cal),
        "FP": int(fp_cal),
        "FN": int(fn_cal),
        "TP": int(tp_cal),
    },

    "predicted_probabilities_calibration": {
        "mean_survived": float(np.mean(y_prob_cal[y_cal == 0])),
        "mean_died": float(np.mean(y_prob_cal[y_cal == 1])),
        "gap_died_minus_survived": float(
            np.mean(y_prob_cal[y_cal == 1]) - np.mean(y_prob_cal[y_cal == 0])
        ),
    },
}

out_path = OUTPUT_DIR / f"Model_5_calibrated_{COHORT_NAME}_{time_stamp}.json"

with open(out_path, "w") as f:
    json.dump(results, f, indent=2)

print("Saved JSON:", out_path)